In [ ]:
import sagemaker
import boto3

sm_client = boto3.client("sagemaker")

# Crear un grupo de modelos en el registry
model_package_group_name = "alquiler-madrid-modelos"

try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Modelos de regresión de precio de alquiler en Madrid"
    )
    print(f"Grupo de modelos '{model_package_group_name}' creado.")
except sm_client.exceptions.ClientError as e:
    print(f"El grupo ya existe o error: {e}")

# Obtener la ruta del mejor modelo del tuning job
tuning_job_name = "sagemaker-scikit-lea-260610-1417"
response = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)
best_job_name = response["BestTrainingJob"]["TrainingJobName"]
best_job = sm_client.describe_training_job(TrainingJobName=best_job_name)
model_artifact = best_job["ModelArtifacts"]["S3ModelArtifacts"]

print(f"Mejor job: {best_job_name}")
print(f"Artefacto del modelo: {model_artifact}")

# Registrar el modelo en el registry
model_package = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="GradientBoosting optimizado con Automatic Model Tuning - R2: 0.7874",
    InferenceSpecification={
        "Containers": [
            {
                "Image": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
                "ModelDataUrl": model_artifact
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"]
    },
    ModelApprovalStatus="Approved"
)

print(f"Modelo registrado con ARN: {model_package['ModelPackageArn']}")

El grupo ya existe o error: An error occurred (ValidationException) when calling the CreateModelPackageGroup operation: Model Package Group already exists: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:model-package-group/alquiler-madrid-modelos
Mejor job: sagemaker-scikit-lea-260610-1417-003-81f656ca
Artefacto del modelo: s3://pablo-proyectoentrega4/models/sagemaker-scikit-lea-260610-1417-003-81f656ca/output/model.tar.gz
Modelo registrado con ARN: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:model-package/alquiler-madrid-modelos/1

In [3]:
import boto3

sm_client = boto3.client("sagemaker")

response = sm_client.list_model_packages(
    ModelPackageGroupName="alquiler-madrid-modelos"
)

for mp in response["ModelPackageSummaryList"]:
    print(f"Versión: {mp['ModelPackageVersion']}")
    print(f"ARN: {mp['ModelPackageArn']}")
    print(f"Estado: {mp['ModelApprovalStatus']}")
    print(f"Creado: {mp['CreationTime']}")

Versión: 1
ARN: arn:aws:sagemaker:us-east-1:214261925557:model-package/alquiler-madrid-modelos/1
Estado: Approved
Creado: 2026-06-10 15:06:31.946000+00:00


In [ ]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

session = sagemaker.Session()

estimator = SKLearn(
    entry_point="train.py",
    role="arn:aws:iam::<ACCOUNT_ID>:role/LabRole",
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    hyperparameters={
        "n-estimators":  150,
        "max-depth":     6,
        "learning-rate": 0.05
    },
    output_path="s3://pablo-proyectoentrega4/models/",
    base_job_name="alquiler-final"
)

estimator.fit({"train": "s3://pablo-proyectoentrega4/processed/"}, wait=True)
print("Entrenamiento completado.")

# Desplegar directamente desde el estimador
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="alquiler-madrid-endpoint"
)
print("Endpoint desplegado correctamente.")

INFO:sagemaker:Creating training-job with name: alquiler-final-2026-06-10-16-04-36-386


2026-06-10 16:04:38 Starting - Starting the training job.

.

.


2026-06-10 16:04:53 Starting - Preparing the instances for training.

...

-

-

-

!

Endpoint desplegado correctamente.


In [18]:
import boto3

runtime = boto3.client("sagemaker-runtime")

# 15 features: floor_area, bedrooms, bathrooms, floor_built, year_built,
# district(0), lift, garage, furnished, terrace, balcony, pool, private, 
# orientation(0), antiguedad
test_data = "80.0,2,1,3.0,1990.0,0,1,0,0,0,0,0,0,0,34.0\n"

response = runtime.invoke_endpoint(
    EndpointName="alquiler-madrid-endpoint",
    ContentType="text/csv",
    Body=test_data
)

resultado = response["Body"].read().decode("utf-8")
print(f"Precio estimado: {resultado} €/mes")

Precio estimado: 1281.226809072972 €/mes


In [ ]:
import sagemaker
import boto3
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

session = sagemaker.Session()

# Crear el monitor
monitor = DefaultModelMonitor(
    role="arn:aws:iam::<ACCOUNT_ID>:role/LabRole",
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

# Generar la línea base (baseline) con los datos procesados
monitor.suggest_baseline(
    baseline_dataset="s3://pablo-proyectoentrega4/processed/",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri="s3://pablo-proyectoentrega4/monitor-baseline/",
    wait=True,
    logs=False
)

print("Baseline generado correctamente.")

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: .


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-10-16-26-07-494


.

.

...

.

.

.

!

Baseline generado correctamente.


In [27]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")

# Crear nueva configuración del endpoint con data capture activado
sm_client.create_endpoint_config(
    EndpointConfigName="alquiler-madrid-endpoint-config-v2",
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": "alquiler-final-2026-06-10-16-07-23-849",
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1,
            "InitialVariantWeight": 1.0
        }
    ],
    DataCaptureConfig={
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": "s3://pablo-proyectoentrega4/data-capture/",
        "CaptureOptions": [
            {"CaptureMode": "Input"},
            {"CaptureMode": "Output"}
        ],
        "CaptureContentTypeHeader": {
            "CsvContentTypes": ["text/csv"]
        }
    }
)

# Actualizar el endpoint con la nueva configuración
sm_client.update_endpoint(
    EndpointName="alquiler-madrid-endpoint",
    EndpointConfigName="alquiler-madrid-endpoint-config-v2"
)

print("Endpoint actualizando con data capture. Esperando...")

waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName="alquiler-madrid-endpoint",
    WaiterConfig={"Delay": 30, "MaxAttempts": 30}
)

print("Endpoint actualizado correctamente con data capture activado.")

Endpoint actualizando con data capture. Esperando...


Endpoint actualizado correctamente con data capture activado.


In [ ]:
import boto3
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator, Statistics, Constraints
import sagemaker

boto_session = boto3.Session(region_name="us-east-1")
session = sagemaker.Session(boto_session=boto_session)

monitor = DefaultModelMonitor(
    role="arn:aws:iam::<ACCOUNT_ID>:role/LabRole",
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

statistics = Statistics.from_s3_uri(
    "s3://pablo-proyectoentrega4/monitor-baseline/statistics.json"
)
constraints = Constraints.from_s3_uri(
    "s3://pablo-proyectoentrega4/monitor-baseline/constraints.json"
)

monitor.create_monitoring_schedule(
    monitor_schedule_name="alquiler-madrid-monitor",
    endpoint_input="alquiler-madrid-endpoint",
    output_s3_uri="s3://pablo-proyectoentrega4/monitor-output/",
    statistics=statistics,
    constraints=constraints,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True
)

print("Schedule de monitorización creado correctamente.")

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: .


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: alquiler-madrid-monitor


Schedule de monitorización creado correctamente.


In [30]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")

response = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName="alquiler-madrid-monitor"
)

print("Nombre:", response["MonitoringScheduleName"])
print("Estado:", response["MonitoringScheduleStatus"])
print("Tipo:", response["MonitoringScheduleConfig"]["MonitoringType"])
print("Expresión cron:", response["MonitoringScheduleConfig"]["ScheduleConfig"]["ScheduleExpression"])

Nombre: alquiler-madrid-monitor
Estado: Scheduled
Tipo: DataQuality
Expresión cron: cron(0 * ? * * *)


In [ ]:
import sagemaker
import boto3
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.parameters import ParameterString

pipeline_session = PipelineSession()

role = "arn:aws:iam::<ACCOUNT_ID>:role/LabRole"
bucket = "pablo-proyectoentrega4"

# Parámetro de entrada
input_data = ParameterString(
    name="InputData",
    default_value=f"s3://{bucket}/processed/"
)

# Paso de entrenamiento
estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    hyperparameters={
        "n-estimators": 150,
        "max-depth": 6,
        "learning-rate": 0.05
    },
    output_path=f"s3://{bucket}/models/",
    base_job_name="pipeline-alquiler",
    sagemaker_session=pipeline_session
)

training_step = TrainingStep(
    name="EntrenarModelo",
    estimator=estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(
            s3_data=input_data,
            content_type="text/csv"
        )
    }
)

# Definir el pipeline
pipeline = Pipeline(
    name="pipeline-reentrenamiento-alquiler",
    parameters=[input_data],
    steps=[training_step],
    sagemaker_session=pipeline_session
)

# Crear el pipeline en AWS
pipeline.upsert(role_arn=role)
print("Pipeline creado correctamente.")

# Ejecutar el pipeline
execution = pipeline.start()
print(f"Pipeline ejecutándose: {execution.arn}")

Pipeline creado correctamente.


Pipeline ejecutándose: arn:aws:sagemaker:us-east-1:214261925557:pipeline/pipeline-reentrenamiento-alquiler/execution/5ev6felwjmhr


In [34]:
import boto3
sm_client = boto3.client("sagemaker")

# Primero borrar el schedule
sm_client.delete_monitoring_schedule(
    MonitoringScheduleName="alquiler-madrid-monitor"
)
print("Schedule eliminado")

# Luego borrar el endpoint
sm_client.delete_endpoint(EndpointName="alquiler-madrid-endpoint")
print("Endpoint eliminado")

Schedule eliminado


ClientError: An error occurred (ValidationException) when calling the DeleteEndpoint operation: The Endpoint currently has one or more MonitoringSchedules. Please delete the MonitoringSchedules before deleting the Endpoint.